## Stream Customers Data from Cloud Files to Delta lake
##### 1. Read Files from clud storage using dataStreamREader API
##### 2. Transform the dataframe to add the following columns
##### - file path: cloud file path
##### -  ingestion date: current timestamp
##### 3. Write the transformed data stream to delta lake Table


### Read files using datastreamreader api

In [0]:
customers_df = (spark.readStream
                .format('cloudfiles')
                .option("cloudFiles.format", "json")              
                .option("cloudFiles.schemaLocation", "/Volumes/gizmobox_catalog_noori/landing/vol_operational_data/Customers_autoloader/_schema")   
                .option("cloudFiles.inferColumnTypes", "true")         
                .option("cloudFiles.schemaHints", "date_of_birth Date, member_since Date, created_timestamp Timestamp") 
                .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                .load("/Volumes/gizmobox_catalog_noori/landing/vol_operational_data/Customers_autoloader/")
)

##### 2. Transform the dataframe to add the following columns
##### - file path: cloud file path
##### -  ingestion date: current timestamp

In [0]:
from pyspark.sql.functions import col, current_timestamp

customer_transform_df = (customers_df.withColumn("filepath", col("_metadata.file_path"))
                    .withColumn("ingestion_date", current_timestamp()))


## 3. Writ the transformed data stream to delta table

In [0]:
streaming_query = (customer_transform_df.writeStream
                    .format("delta")
                    .option("checkpointLocation", "/Volumes/gizmobox_catalog_noori/landing/vol_operational_data/Customers_autoloader/_checkpoint_stream")
                    .option("mergeSchema", "true")
                    .toTable("gizmobox_catalog_noori.bronze.customer_stream_autoloader")
)

In [0]:
streaming_query.stop()

In [0]:
%sql
drop table gizmobox_catalog_noori.bronze.customer_stream_autoloader

In [0]:
%sql
select * from gizmobox_catalog_noori.bronze.customer_stream_autoloader